Librerias

In [ ]:
RandomForestRegressor / RandomForestClassifier

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. REPRODUCIBILIDAD Y CARGA DE DATOS

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

# 2. PROCESAMIENTO E INGENIERÍA DE VARIABLES

In [3]:
def preprocess_features(df):
    df = df.copy()
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df[spend_cols] = df[spend_cols].fillna(0)
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    df['HasSpented'] = (df['TotalSpend'] > 0).astype(str)
    
    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0] if pd.notnull(x) else '0000')
    group_sizes = df['Group'].value_counts()
    df['GroupSize'] = df['Group'].map(group_sizes)
    df['IsAlone'] = (df['GroupSize'] == 1).astype(str)
    
    df['Cabin'] = df['Cabin'].fillna('U/0/U')
    df['Cabin_Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
    df['Cabin_Side'] = df['Cabin'].apply(lambda x: x.split('/')[-1])
    return df

train_processed = preprocess_features(train_df)
test_processed = preprocess_features(test_df)

num_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend', 'GroupSize']
cat_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Cabin_Deck', 'Cabin_Side', 'HasSpented', 'IsAlone']

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

X = train_processed[num_features + cat_features]
y = train_processed['Transported'].astype(int).values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_trans = preprocessor.fit_transform(X_train)
X_val_trans = preprocessor.transform(X_val)
X_test_trans = preprocessor.transform(test_processed[num_features + cat_features])

# 3. CONSTRUCCIÓN DE LA RED NEURONAL RESIDUAL (API FUNCIONAL)

In [4]:
print("--- CONSTRUYENDO MODELO RESIDUAL (TABULAR RESNET) ---")

input_dim = X_train_trans.shape[1]
inputs = layers.Input(shape=(input_dim,), name="Input_Features")

# Capa de proyección inicial (Mapea las entradas a una dimensión intermedia)
x = layers.Dense(128, activation='relu')(inputs)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

# --- BLOQUE RESIDUAL 1 ---
# Rama profunda
res1 = layers.Dense(128, activation='relu')(x)
res1 = layers.BatchNormalization()(res1)
res1 = layers.Dropout(0.3)(res1)
res1 = layers.Dense(128, activation='relu')(res1)
res1 = layers.BatchNormalization()(res1)
# Conexión de salto (Sumamos la entrada 'x' a la salida 'res1')
x = layers.add([x, res1]) 

# --- BLOQUE RESIDUAL 2 ---
# Rama profunda (reducimos la dimensión a 64)
x_proj = layers.Dense(64, activation='relu')(x) # Proyección para que coincidan las dimensiones al sumar
x_proj = layers.BatchNormalization()(x_proj)

res2 = layers.Dense(64, activation='relu')(x_proj)
res2 = layers.BatchNormalization()(res2)
res2 = layers.Dropout(0.3)(res2)
res2 = layers.Dense(64, activation='relu')(res2)
res2 = layers.BatchNormalization()(res2)
# Conexión de salto
x = layers.add([x_proj, res2])

# --- CAPA DE SALIDA ---
x = layers.Dense(32, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(1, activation='sigmoid', name="Output_Layer")(x)

# Instanciamos el modelo definiendo entradas y salidas
model = keras.Model(inputs=inputs, outputs=outputs, name="Tabular_ResNet")

model.summary() # Esto imprimirá la estructura y mostrará visualmente las conexiones cruzadas

--- CONSTRUYENDO MODELO RESIDUAL (TABULAR RESNET) ---


Model: "Tabular_ResNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_Features      │ (None, 34)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │      4,480 │ Input_Features[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128)       │        512 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128)       │          0 │ dropout[0][0],    │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_3[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      4,160 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_4[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │      4,160 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_5[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 64)        │          0 │ batch_normalizat… │
│                     │                   │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │      2,080 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense_6[0][0]     │
│ (BatchNormalizatio… │                   │            │                 

 Total params: 58,625 (229.00 KB)

 Trainable params: 57,409 (224.25 KB)

 Non-trainable params: 1,216 (4.75 KB)

# 4. ENTRENAMIENTO

In [5]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.003),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Callbacks para evitar sobreajuste y optimizar el aprendizaje
lr_decay = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5, 
    patience=3, 
    verbose=1,
    min_lr=1e-5
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_trans, y_train,
    validation_data=(X_val_trans, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[early_stopping, lr_decay],
    verbose=1
)

Epoch 1/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7433 - loss: 0.5200 - val_accuracy: 0.7901 - val_loss: 0.4347 - learning_rate: 0.0030
Epoch 2/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7773 - loss: 0.4507 - val_accuracy: 0.7855 - val_loss: 0.4204 - learning_rate: 0.0030
Epoch 3/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7908 - loss: 0.4386 - val_accuracy: 0.7855 - val_loss: 0.4220 - learning_rate: 0.0030
Epoch 4/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7896 - loss: 0.4334 - val_accuracy: 0.7844 - val_loss: 0.4186 - learning_rate: 0.0030
Epoch 5/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7925 - loss: 0.4284 - val_accuracy: 0.7826 - val_loss: 0.4201 - learning_rate: 0.0030
Epoch 6/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7985 - loss: 0.4223 - val_accuracy: 0.7895 - val_loss: 0.4137 - learning_rate: 0.0030
Epoch 7/60
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8014 - loss: 0.4206 - 

# 5. GENERACIÓN DEL ARCHIVO SUBMISSION_4.CSV

In [6]:
print("\n--- GENERANDO SUBMISSION_4.CSV ---")
test_preds_proba = model.predict(X_test_trans)
test_preds_boolean = (test_preds_proba > 0.5).astype(bool).flatten()

submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': test_preds_boolean
})

submission_df.to_csv('submission_4.csv', index=False)
print("✅ ¡Archivo 'submission_4.csv' guardado exitosamente!")
print(submission_df['Transported'].value_counts())


--- GENERANDO SUBMISSION_4.CSV ---
134/134 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ ¡Archivo 'submission_4.csv' guardado exitosamente!
Transported
True     2179
False    2098
Name: count, dtype: int64
